In [0]:
# =====================================
# NOTEBOOK : NB_SILVER_MEMBER
# LAYER    : SILVER
# TARGET   : ODS_MEMBER
# =====================================


In [0]:
%run ../COMMON/NB_CONFIG

In [0]:
%run ../COMMON/NB_CONNECTION

In [0]:
%run ../COMMON/NB_UTILS

In [0]:
member_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.MEMBER",
    properties=connectionProperties
)

address_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.ADDRESS",
    properties=connectionProperties
)

print("MEMBER COUNT :", member_df.count())
print("ADDRESS COUNT:", address_df.count())

In [0]:
null_count = member_df.filter(
    F.col("MEMBER_ID").isNull()
).count()

if null_count > 0:
    raise Exception(
        f"NULL MEMBER_ID FOUND : {null_count}"
    )

In [0]:
joined_df = (
    member_df.alias("m")
    .join(
        address_df.alias("a"),
        F.col("m.MEMBER_ID") == F.col("a.MEMBER_ID"),
        "left"
    )
)

In [0]:
ods_member_df = joined_df.select(
    F.col("m.MEMBER_ID"),
    F.col("m.FIRST_NAME"),
    F.col("m.LAST_NAME"),
    F.col("m.DOB"),
    F.col("m.GENDER"),
    F.col("m.PLAN_ID"),
    F.col("m.EMAIL"),
    F.col("m.PHONE"),

    F.col("a.ADDRESS_LINE1"),
    F.col("a.ADDRESS_LINE2"),
    F.col("a.CITY"),
    F.col("a.STATE"),
    F.col("a.ZIP_CODE"),
    F.col("a.COUNTRY"),

    F.col("m.SRC_CREATED_DATETIME")
)

In [0]:
ods_member_df = (
    ods_member_df

    .withColumn(
        "FULL_NAME",
        F.initcap(
            F.concat_ws(
                " ",
                F.col("FIRST_NAME"),
                F.col("LAST_NAME")
            )
        )
    )

  .withColumn(
    "AGE",
    F.floor(
        F.datediff(
            F.current_date(),
            F.col("DOB")
        ) / 365
    ).cast("int")
)
    .withColumn(
        "AGE_GROUP",
        F.when(
            F.col("AGE") < 18,
            "CHILD"
        )
        .when(
            F.col("AGE") < 60,
            "ADULT"
        )
        .otherwise(
            "SENIOR"
        )
    )
)

In [0]:
ods_member_df = ods_member_df.fillna(
    {
        "CITY":"UNKNOWN",
        "STATE":"UNKNOWN",
        "COUNTRY":"INDIA",
        "ZIP_CODE":"000000"
    }
)

In [0]:
window_spec = Window.partitionBy(
    "MEMBER_ID"
).orderBy(
    F.col("MEMBER_ID")
)

ods_member_df = (
    ods_member_df
    .withColumn(
        "RN",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("RN") == 1
    )
    .drop("RN")
)

In [0]:
ods_member_df = ods_member_df.withColumn(
    "HASH_VALUE",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("FIRST_NAME"),
            F.col("LAST_NAME"),
            F.col("DOB"),
            F.col("GENDER"),
            F.col("PLAN_ID"),
            F.col("EMAIL"),
            F.col("PHONE"),
            F.col("ADDRESS_LINE1"),
            F.col("CITY"),
            F.col("STATE"),
            F.col("ZIP_CODE"),
            F.col("COUNTRY")
        ),
        256
    )
)

In [0]:
ods_member_df = (
    ods_member_df

    .withColumn(
        "LOAD_DATE",
        F.current_timestamp()
    )

    .withColumn(
        "IS_CURRENT",
        F.lit("Y")
    )

    .withColumn(
        "EFFECTIVE_DATE",
        F.current_timestamp()
    )

    .withColumn(
        "END_DATE",
        F.lit("9999-12-31")
        .cast("timestamp")
    )
)

In [0]:
target_count = spark.table(ods_member_table).count()
print(f"TARGET RECORD COUNT : {target_count}")

if target_count == 0:
    # Initial full load (MEMBER_HIST_KEY auto-generated by Delta identity column)
    ods_member_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ods_member_table)

    print("INITIAL LOAD COMPLETED")

    # Exit notebook so incremental logic does not run
    dbutils.notebook.exit("Initial load done")

else:
    # Incremental load (SCD2)
    print("SCD2 LOAD STARTED")

    target_df = spark.table(ods_member_table)

    current_df = target_df.filter(F.col("IS_CURRENT") == "Y")
    print(f"CURRENT ACTIVE RECORDS : {current_df.count()}")

In [0]:
changed_df = (
        ods_member_df.alias("s")
        .join(
            current_df.alias("t"),
            "MEMBER_ID"
        )
        .filter(F.col("s.HASH_VALUE") != F.col("t.HASH_VALUE"))
        .select("s.*")
    )

In [0]:
new_member_df = (
    ods_member_df.alias("s")
    .join(
        current_df.alias("t"),
        "MEMBER_ID",
        "left_anti"
    )
)

In [0]:
delta_table = DeltaTable.forName(spark, ods_member_table)

expire_df = changed_df.select("MEMBER_ID").distinct()

delta_table.alias("t").merge(
    expire_df.alias("s"),
    "t.MEMBER_ID = s.MEMBER_ID AND t.IS_CURRENT = 'Y'"
).whenMatchedUpdate(
    set={
        "IS_CURRENT": "'N'",               # mark old record as not current
        "END_DATE": "current_timestamp()"  # set end date to now
    }
).execute()


In [0]:
changed_insert_df = (
        changed_df
        .withColumn("LOAD_DATE", F.current_timestamp())
        .withColumn("IS_CURRENT", F.lit("Y"))
        .withColumn("EFFECTIVE_DATE", F.current_timestamp())
        .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
    )

changed_insert_df.write.format("delta").mode("append").saveAsTable(ods_member_table)

In [0]:
new_member_df = (
    new_member_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

new_member_df.write.format("delta").mode("append").saveAsTable(ods_member_table)

In [0]:
display(
    spark.sql(f"""
    SELECT
        *
    FROM {ods_member_table}
    ORDER BY MEMBER_ID
    """)
)